## Create variable for number of conflicts in previous month

* __input__:
    + import `Dataset/create_acs_conflict/acs_conflict.csv` for number of conflicts in 2022
    + import `2_Dataset/b_Partial_Datasets/joined_ACLED_ZIP_21_PA.csv` for number of conflicts in 2021 --> only need dec. 2021

* __process__:
    + within `df_events_2021`, create `event_count` variable for dec. 2021
    + create `df_events_all`: merge dec. 2021 observations from `df_events_2021` onto `df_events_2022`
    + within `df_events_all`, create `event_count_prev` for conflict in previous month

* __output__:
    + `df_events_all`:
        + contains all obs. per zip code and month (21996)
        + `event_count`: number of conflicts per zip code per month
        + `event_count_prev`: number of conflicts per zip code in _previous_ month (** using data from dec. 2021)

* __saved as__:
    + `1_Features/b_Conflict_Features/Conflict_PreviousMonth/acs_conflict_previousmonth.csv`
    + `Dataset/create_acs_conflict/acs_conflict_previousmonth.csv`
    + `2_Dataset/b_Partial_Datasets/ACLED_conflict_previousmonth_allZCTA.csv`

<br>
<hr>

### 0. Setup

In [1]:
# dynamic search path configuration
import sys
import os

# adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

# adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

Current Project Root Directory set to: /Users/varvarailyina/hertie/ML/US_conflict_forecasting/
Current Search Path set to: ['/Users/varvarailyina/hertie/ML/US_conflict_forecasting', '/Users/varvarailyina/hertie/ML/US_conflict_forecasting/Dataset', '/Users/varvarailyina/miniconda3/envs/ML_lab/lib/python311.zip', '/Users/varvarailyina/miniconda3/envs/ML_lab/lib/python3.11', '/Users/varvarailyina/miniconda3/envs/ML_lab/lib/python3.11/lib-dynload', '', '/Users/varvarailyina/miniconda3/envs/ML_lab/lib/python3.11/site-packages', '/Users/varvarailyina/hertie/ML/US_conflict_forecasting/']


In [2]:
# import necessary packages
import numpy as np
import pandas as pd
import os

<br>
<hr>
<br>

### 1. Calculate number of conflicts in December 2021

In [3]:
# load acled zip 2021 data
acled_21 = pd.read_csv(os.path.join(project_root_dir, './2_Dataset/b_Partial_Datasets/joined_ACLED_ZIP_21_PA.csv'))

# remove unnecessary vars
df_events_2021 = acled_21[['event_id_cnty', 'event_date', 'year', 'ZCTA5CE10']].copy()

# convert date var to YYYY-MM format
df_events_2021.loc[:, 'parsed_date'] = pd.to_datetime(df_events_2021['event_date'], format = '%d %B %Y')
df_events_2021['Month'] = df_events_2021['parsed_date']
df_events_2021['Month'] = df_events_2021['Month'].dt.strftime('%Y-%m')

# filter for december 2021
df_events_2021 = df_events_2021.loc[df_events_2021['Month'] == '2021-12']

# prepare for merging
df_events_2021 = df_events_2021[['event_id_cnty', 'Month', 'ZCTA5CE10']]
df_events_2021 = df_events_2021.rename(columns = {'ZCTA5CE10': 'ZCTA'})

# order the data
df_events_2021 = df_events_2021.sort_values(by = 'ZCTA')

# create `event_count` variable
df_events_2021['event_count'] = df_events_2021.groupby('ZCTA')['ZCTA'].transform('count')

# remove unnecessary vars
df_events_2021 = df_events_2021.drop(columns=['event_id_cnty'])

# make df unique
df_events_2021 = df_events_2021.drop_duplicates()

# peek into df
print(df_events_2021.head(), '\n')
print(df_events_2021.shape)

       Month   ZCTA  event_count
168  2021-12  15219            6
261  2021-12  16501            2
352  2021-12  16801            1
281  2021-12  17102            3
377  2021-12  17602            1 

(9, 3)


<hr>
<br>

### 2. Merge observations for December 2021 onto conflict data

#### 2.1 prepare acs 2022 conflict data for merging

In [4]:
# load conflict data
acs_2022 = pd.read_csv("./Dataset/create_acs_conflict/acs_conflict.csv")

# remove unnecessary vars
df_events_2022 = acs_2022[['ZCTA', 'Month', 'event_count']].copy()

# sort df by zip code and month
df_events_2022 = df_events_2022.sort_values(by = ['ZCTA', 'Month'])

#### 2.2 create empty row for december 2021 in 2022 conflict data

In [5]:
# new row with `2021-12` per zip code
new_rows = []
for group_name, group_data in df_events_2022.groupby('ZCTA'):
    new_row = {'ZCTA': group_name, 'Month': '2021-12', 'event_count': 0}
    new_rows.append(new_row)

# make df from new_rows
df_new_rows = pd.DataFrame(new_rows)

# concatenate `df_new_rows` with `df_events_2022`
df_events_all = pd.concat([df_events_2022, df_new_rows], ignore_index = True)

# sort by zip code and month
df_events_all = df_events_all.sort_values(by = ['ZCTA', 'Month'])

# peek into df
print(df_events_all.head(), '\n')
print(df_events_all.shape)


          ZCTA    Month  event_count
21996  15001.0  2021-12          0.0
0      15001.0  2022-01          1.0
1      15001.0  2022-02          0.0
2      15001.0  2022-03          0.0
3      15001.0  2022-04          0.0 

(23829, 3)


#### 2.3 merge number of conflicts in december 2021 into 2022 data

In [6]:
# input actual `event_count` for december 2021 from `df_events_2021` with suffix `_new`
df_events_all = pd.merge(df_events_all, df_events_2021, on = ['ZCTA', 'Month'], how = 'left', suffixes = ('', '_new'))

# fill missing 'event_count_new' values with 0
df_events_all['event_count_new'] = df_events_all['event_count_new'].fillna(0)

# update 'event_count' column with actual values only for Month = '2021-12'
df_events_all.loc[df_events_all['Month'] == '2021-12', 'event_count'] = df_events_all['event_count_new']

# drop 'event_count_new' column
df_events_all.drop(columns=['event_count_new'], inplace = True)

# peek into df
print(df_events_all.head(), '\n')
print(df_events_all.shape)

      ZCTA    Month  event_count
0  15001.0  2021-12          0.0
1  15001.0  2022-01          1.0
2  15001.0  2022-02          0.0
3  15001.0  2022-03          0.0
4  15001.0  2022-04          0.0 

(23829, 3)


<hr>
<br>

### 3. Create variable for conflict in previous month

In [7]:
# create `event_count_prev` for number of events in previous month by zip code
df_events_all['event_count_prev'] = df_events_all.groupby('ZCTA')['event_count'].shift()

# remove rows for december 2021
df_events_all = df_events_all[df_events_all['Month'] != '2021-12']

# reset index
df_events_all.reset_index(drop = True, inplace = True)

# peek into df
print(df_events_all.head(), '\n')
print(df_events_all.shape)

      ZCTA    Month  event_count  event_count_prev
0  15001.0  2022-01          1.0               0.0
1  15001.0  2022-02          0.0               1.0
2  15001.0  2022-03          0.0               0.0
3  15001.0  2022-04          0.0               0.0
4  15001.0  2022-05          0.0               0.0 

(21996, 4)


In [8]:
# TEST: look at zip codes with repeated conflicts
df_events_all[(df_events_all['event_count'] > 0) & (df_events_all['event_count_prev'] > 0)]

,ZCTA,Month,event_count,event_count_prev
1404,15219.0,2022-01,5.0,6.0
1405,15219.0,2022-02,10.0,5.0
1406,15219.0,2022-03,10.0,10.0
1407,15219.0,2022-04,10.0,10.0
1408,15219.0,2022-05,8.0,10.0
...,...,...,...,...
21029,19406.0,2022-06,2.0,1.0
21255,19464.0,2022-04,1.0,1.0
21879,19601.0,2022-04,1.0,2.0
21880,19601.0,2022-05,1.0,1.0


<hr>
<br>

### 4. Save updated dataset

In [12]:
# save df where needed
df_events_all.to_csv("./1_Features/b_Conflict_Features/Conflict_PreviousMonth/acs_conflict_previousmonth.csv", index = False)

df_events_all.to_csv("./Dataset/create_acs_conflict/acs_conflict_previousmonth.csv", index = False)

df_events_all.to_csv("./2_Dataset/b_Partial_Datasets/ACLED_conflict_previousmonth_allZCTA.csv", index = False)